# 03 — Modeling

**Goal:** Train multiple classifiers and compare.

Models to try:
1. Logistic Regression (linear baseline)
2. KNN (instance-based)
3. SVM (margin-based)
4. Decision Tree (rule-based)
5. Random Forest (ensemble)

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

from src.data_loader import load_raw_data
from src.preprocess import split_and_scale
from src.train import train_models

df = load_raw_data(save_csv=False)
X_train, X_test, y_train, y_test, scaler, feature_names = split_and_scale(df)

## Train All Models

We use `src/train.py` to keep this reusable.

In [ ]:
models = train_models(X_train, y_train, save=True)

## Quick Accuracy Comparison

In [ ]:
import pandas as pd

results = {name: m.score(X_test, y_test) for name, m in models.items()}
results_df = pd.DataFrame.from_dict(results, orient="index", columns=["accuracy"])
results_df.sort_values("accuracy", ascending=False)

## 5-Fold Cross-Validation

A single split can be misleading. CV gives a more robust estimate.

In [ ]:
from sklearn.model_selection import cross_val_score

for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=5)
    print(f"{name:15s} | {scores.mean():.4f} ± {scores.std():.4f}")

**Observation:** SVM & Logistic Regression are top performers.

## Hyperparameter Tuning (SVM)

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC
from src.config import SVM_GRID

grid = GridSearchCV(SVC(probability=True), SVM_GRID, cv=5, scoring="accuracy")
grid.fit(X_train, y_train)

print("Best params:", grid.best_params_)
print("Best CV score:", round(grid.best_score_, 4))
print("Test score :", round(grid.score(X_test, y_test), 4))

## Save Best Model

In [ ]:
import joblib
from src.config import MODELS_DIR

joblib.dump(grid.best_estimator_, MODELS_DIR / "svm_tuned.pkl")
print("✅ Saved tuned SVM")

**Next:** → `04_evaluation.ipynb`